# Конспект: валидация, кросс-валидация и верификация

## Глава 1. Валидация и кросс-валидация


### 1) Что такое валидация (validation)

**Валидация** — это проверка качества модели на данных, которые **не участвовали в обучении**, но относятся к той же задаче и распределению.

В практическом ML под «валидацией» чаще всего понимают один из вариантов:
- **Hold-out validation**: одно разбиение `train/val` (например, 80/20).
- **Cross-validation**: многократные разбиения (k-fold и варианты).

Ключевой смысл: на валидации мы оцениваем **обобщающую способность** модели и принимаем решения про:
- выбор признаков/фич,
- выбор семейства модели,
- подбор гиперпараметров,
- раннюю остановку (early stopping) и т.п.


### 2) Что такое кросс-валидация (cross-validation)

**Кросс-валидация** — это подход, при котором данные разбиваются на `K` частей (**folds**), и модель обучается `K` раз: каждый раз одна часть служит валидацией, остальные — обучением.

В результате ты получаешь:
- **K значений метрики**,
- их **среднее** (оценка качества),
- и часто **разброс** (стабильность/устойчивость).

Частые варианты:
- **KFold** — обычный k-fold.
- **StratifiedKFold** — для классификации, сохраняет доли классов в каждом fold.
- **GroupKFold** — когда есть группы (например, пользователи/пациенты), и нельзя смешивать группы между train/val.
- **TimeSeriesSplit** — для временных рядов (нельзя «смотреть в будущее»).


### 3) Когда использовать hold-out (одно разбиение), а когда k-fold

**Hold-out (`train/val split`) обычно достаточно, если:**
- данных много,
- обучение дорогое/долгое,
- нужно быстро тестировать идеи,
- данные однородные и разбиение можно сделать репрезентативно.

**K-fold CV лучше, если:**
- данных мало или умеренно,
- качество сильно зависит от конкретного разбиения,
- нужна более надёжная оценка (и оценка разброса),
- ты делаешь серьёзный подбор гиперпараметров.


### 4) Типичная схема «train/val/test»

Частая и удобная практика:
1. **Train** — обучаем модель.
2. **Validation** — выбираем гиперпараметры/модель/фичи.
3. **Test** — *один раз* оцениваем итоговое качество (после выбора всех решений).

Почему так важно иметь test:
- если ты много раз пробовал разные модели/фичи/параметры и смотрел на validation, то постепенно ты **подгоняешься под validation**;
- test нужен как «последний честный экзамен».


### 5) Примеры в sklearn

#### 5.1 Hold-out: одно разбиение `train/val`
```python
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # полезно для классификации
)
```

#### 5.2 K-fold CV: оценка качества
```python
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print('CV scores:', scores)
print('Mean:', scores.mean())
print('Std:', scores.std())
```


### 6) Подбор гиперпараметров: почему обычно используют кросс-валидацию

Почему кросс-валидация используется для подбора гиперпараметров

1. Стабильность оценки
Модель тестируется на нескольких подвыборках, и метрика усредняется. Это делает результат более надёжным.

2. Эффективное использование данных
Ты используешь все данные как в train, так и в validation в разных разбиениях. Особенно ценно, если данных немного.

3. Устойчивость к переобучению 
Ты не оптимизируешь модель на одной и той же валидации, а как бы "усредняешь" по разным сценариям — это снижает риск подгонки гиперпараметров под конкретную подвыборку.


#### Расширение: что именно даёт CV при подборе гиперпараметров

Когда ты подбираешь гиперпараметры (например, `max_depth`, `C`, `learning_rate`), ты по сути проводишь **поиск по множеству конфигураций**. Если оценка качества опирается на одно разбиение, ты рискуешь:
- выбрать параметры, которые «повезло» попасть на удачный split,
- или наоборот отвергнуть хорошие параметры из-за неудачной валидации.

Кросс-валидация уменьшает этот риск, потому что:
- усредняет качество по нескольким валид-подвыборкам;
- делает выбор параметров менее зависимым от одного конкретного split.


#### Пример: GridSearchCV / RandomizedSearchCV

```python
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X, y)
print('best params:', grid.best_params_)
print('best CV score:', grid.best_score_)
```

Важное: `best_score_` — это **средняя метрика по folds** для лучшей конфигурации.


### 7) Важные нюансы и типичные ошибки

1) **Data leakage (утечка данных)**
- Нельзя делать масштабирование/кодирование/импутацию на всём датасете до split/CV.
- Правильно: использовать `Pipeline`, чтобы трансформации обучались только на train-fold.

2) **Не тот CV-сплиттер**
- Классификация с дисбалансом: чаще нужен `StratifiedKFold`.
- Если есть группы (например, один пользователь даёт много строк): нужен `GroupKFold`.
- Временные ряды: нельзя перемешивать; нужен `TimeSeriesSplit`.

3) **Nested cross-validation (когда нужно)**
- Если ты хочешь *честно* оценить качество при активном подборе гиперпараметров, иногда используют вложенную CV:
  - внешняя CV оценивает качество,
  - внутренняя CV подбирает гиперпараметры.
- Это дорого, но даёт более объективную оценку, особенно на маленьких датасетах.


## Глава 2. Валидация и верификация


### 1) Определения (чётко)

**Верификация (verification)** — проверка, что ты **реализовал систему правильно**: код, логика, пайплайн, формулы, отсутствие утечек, корректность разбиений.

**Валидация (validation)** — проверка, что система/модель **решает нужную задачу достаточно хорошо** на новых данных (обычно размеченных).

Классическая формулировка:
- **Verification:** *Did we build the system right?* (Мы построили систему правильно?)
- **Validation:** *Did we build the right system?* (Мы построили правильную систему?)


### 2) В контексте ML-проекта

**Верификация** — это про техническую корректность:
- правильно ли сделан split (`train/val/test`, группы, время);
- нет ли утечки таргета в признаки;
- корректно ли работает preprocessing;
- правильно ли считаются метрики;
- воспроизводимость (random_state/seed, фиксированные версии библиотек);
- тесты/проверки на NaN, формы массивов, типы.

**Валидация** — это про качество:
- метрика на validation/test;
- устойчивость качества (CV-распределение метрики);
- анализ ошибок (какие кейсы валятся);
- сравнение с бейзлайном.


### 3) Пример (из твоего конспекта) + разбор


Пример<br>
Проект: Прогноз риска сердечного заболевания<br>
Цель: Обучить модель, которая по медицинским признакам предсказывает, есть ли у пациента риск болезни (бинарная классификация: 0 или 1).


#### Как это выглядит на практике в этом проекте

**Верификация (технические проверки):**
1) Проверить, что `target` не попал в признаки.
2) Проверить отсутствие утечек: например, признаки, рассчитанные из будущих данных, не должны попадать в train.
3) Проверить корректность предобработки:
   - импутация NaN,
   - масштабирование,
   - кодирование категорий.
4) Проверить корректность разбиения:
   - стратификация классов (если надо),
   - групповой split (если есть несколько записей на одного человека),
   - временной split (если данные по времени).

**Валидация (оценка качества):**
1) Обучить на train.
2) Оценить на validation (или CV).
3) Выбрать модель/порог/гиперпараметры.
4) Финально оценить на test.


### 4) Примеры верификационных проверок (assert / sanity checks)

```python
# 1) Проверки на NaN
assert not X_train.isnull().any().any(), 'В X_train есть NaN!'
assert not y_train.isnull().any(), 'В y_train есть NaN!'

# 2) Проверка, что таргет не в признаках
assert 'target' not in X_train.columns, 'Утечка: target попал в признаки!'

# 3) Проверка размеров
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
```

Такие проверки — это типичная часть **верификации**: они помогают быстро поймать грубые ошибки в данных и пайплайне.


### 5) Примеры валидации (оценка качества)

```python
from sklearn.metrics import accuracy_score, f1_score

model.fit(X_train, y_train)
pred = model.predict(X_val)

print('Accuracy:', accuracy_score(y_val, pred))
print('F1:', f1_score(y_val, pred))
```

Это — **валидация**: мы измеряем качество на данных, которые не участвовали в обучении.


### 6) Где здесь юнит-тесты

Юнит-тесты — это инструмент, который обычно относится к **верификации**:
- проверяют отдельные функции (например, `preprocess()`, `feature_engineering()`, `predict()`),
- ловят ошибки после изменений,
- помогают поддерживать проект, когда кода становится много.

Пример (pytest):
```python
def test_preprocess_no_nan():
    Xp = preprocess(X_raw)
    assert not Xp.isnull().any().any()
```
